[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kjibran/euramet_ml_noise_workshop/blob/main/01_noise_calibration.ipynb)

# Noise Sensors: Calibration, Features and Data Split

Here you calibrate a low-cost node step by step and find out which **features** and which **data split** give a
model you can trust.

```text
 node ──┐
        ├─> clean ──> features X ──> split ──> fit ──> score ──> calibrated
 ref ───┘   + fix      + target y     train     model   RMSE      LAeq
            clock                     + test            MBE, R²
```

**Calibration period**: in practice a node is co-located with a reference for a few weeks and then moved into the
network. Here the first four weeks (1 to 28 September) play that role.

---

# 🧩 Task 2: Which features, which model and which split would you choose?

📝 Note your choices, each with a reason, in the template.

---

# 0 Python Setup and Interface

## 0.0 Notebook Interaction

- **Cells with code**: select the cell and press `Shift`+`Enter` to run it and move on, or `Ctrl`+`Enter` to run it and stay
- **Symbols before code cells**:
  - **▶️ Run**: run the cell as it is
  - **🚀 Action**: run it, change a value, run it again
- \# 💬 **Comment**: Python skips everything after `#`
- Run the cells **from top to bottom**. If something breaks: *Kernel* → *Restart & Run All*.

In [ ]:
# ▶️ Run: get the data (needed on Google Colab, skipped on your own computer)
import os, sys, zipfile
from pathlib import Path

REPO = "euramet_ml_noise_workshop"
REPO_URL = f"https://github.com/kjibran/{REPO}.git"

if not Path("data").exists() and "google.colab" in sys.modules:
    folder = REPO
    if not Path(folder, "data").exists():
        os.system(f"git clone -q {REPO_URL}")
    if not Path(folder, "data").exists():
        from google.colab import files
        print("Download failed. Please upload ml_noise_workshop.zip (from your tutor).")
        zipfile.ZipFile(next(iter(files.upload()))).extractall(".")
        folder = "ml_noise_workshop"
    os.chdir(folder)
print("Data folder found:", Path("data").exists())

In [ ]:
# ▶️ Run
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_predict

%matplotlib inline
plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["date.converter"] = "concise"  # short date labels

## 0.1 Helper Functions

They load the data as in notebook 00, fix the clock of `N1`, join reference and weather, and score a model.
No need to read them, but feel free to.

##### ▶️ Run

In [ ]:
# ▶️ Run
FOLDER = Path("./data")
NODES = ["N1", "N2", "N3"]
COLORS = {"N1": "#2a78d6", "N2": "#1baf7a", "N3": "#eb6834", "Reference": "#0b0b0b"}

ref = pl.read_csv(FOLDER / "reference_class1.csv", try_parse_dates=True).rename(
    {"time_utc": "time", "laeq_10min": "ref_laeq"})
meteo = pl.read_csv(FOLDER / "meteo.csv", try_parse_dates=True).rename({"time_utc": "time"})
nodes = pl.concat(
    pl.read_csv(FOLDER / f"node_{node}.csv", try_parse_dates=True).select(pl.lit(node).alias("ID"), pl.all())
    for node in NODES
).sort("ID", "time")


def energy_mean(column: str) -> pl.Expr:
    """Average levels in energy: 10·log10(mean(10^(L/10))). Never take the plain mean of dB values."""
    return 10 * (10 ** (pl.col(column) / 10)).mean().log10()

In [ ]:
# ▶️ Run

def fix_clock(nodes: pl.DataFrame) -> pl.DataFrame:
    """N1 logs local time in Bern (UTC+2 in summer, UTC+1 after 26 Oct). Convert it to UTC like the others.
    The repeated hour on 26 Oct cannot be resolved, so those readings are dropped."""
    n1 = nodes.filter(pl.col("ID") == "N1").with_columns(
        pl.col("time").dt.replace_time_zone("Europe/Zurich", ambiguous="null")
        .dt.convert_time_zone("UTC").dt.replace_time_zone(None)
    ).drop_nulls("time")
    return pl.concat([n1, nodes.filter(pl.col("ID") != "N1")]).sort("ID", "time")

def build_table(nodes: pl.DataFrame) -> pl.DataFrame:
    """One row per node and 10 minutes: node reading, reference, weather. Rows without a reference value are dropped."""
    return (
        fix_clock(nodes)
        .join(ref, on="time", how="left")
        .join(meteo, on="time", how="left")
        .drop_nulls(["laeq_10min", "ref_laeq"])
        .with_columns(
            hour=pl.col("time").dt.hour(),
            day=(pl.col("time") - pl.datetime(2025, 9, 1)).dt.total_minutes() / 1440,
        )
    )

def scores(y_true, y_pred) -> dict:
    """R², RMSE [dB], MBE [dB] and the share of readings within ±2 dB of the reference."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    err = y_pred - y_true
    return {
        "R²": round(float(r2_score(y_true, y_pred)), 3),
        "RMSE": round(float(np.sqrt(np.mean(err ** 2))), 2),
        "MBE": round(float(np.mean(err)), 2),
        "within ±2 dB": f"{100 * np.mean(np.abs(err) <= 2):.0f} %",
    }

data = build_table(nodes)
CAL_END = dt.datetime(2025, 9, 29)  # end of the co-location (calibration) period
print(data.group_by("ID").agg(pl.len().alias("rows")).sort("ID"))

## 1.0 Features and Target

- **Target `y`**: what the model should predict, the reference `ref_laeq`.
- **Features `X`**: what the model may use, everything the node and the weather station report.

We start with one node.

##### ▶️ Run

In [ ]:
SENSOR = "N1"  # used below: try "N2", "N3"

d = data.filter(pl.col("ID") == SENSOR)
CANDIDATES = ["laeq_10min", "wind_speed", "rain_mm", "temperature", "relative_humidity", "hour"]
print(f"{SENSOR}: {d.height} rows")
d.select(["time", "ref_laeq"] + CANDIDATES).head()

## 1.1 📈 A Straight Line

The simplest calibration is `reference ≈ a × node + b`. Every reading is predicted once by a model that did not
see it (5-fold cross-validation, shuffled rows).

Scores, all in dB except R²:
- **RMSE**: the typical error, large misses count extra
- **MBE**: the systematic offset, + means the calibrated node reads too high
- **within ±2 dB**: the share of readings close enough for most network uses

##### ▶️ Run

In [ ]:
# ▶️ Run
def plot_vs_reference(y, raw, cal, title):
    fig, ax = plt.subplots(figsize=(6, 5.5))
    ax.scatter(y, raw, s=3, alpha=0.3, color="#9FA1A4", label="raw node")
    ax.scatter(y, cal, s=3, alpha=0.4, color="#174A7C", label="calibrated")
    ax.plot([30, 75], [30, 75], "k--", lw=1)
    ax.set_xlabel("Reference LAeq,10min (dB)"); ax.set_ylabel("Node LAeq,10min (dB)"); ax.set_title(title)
    ax.legend(markerscale=4); plt.show()

X, y = d.select(["laeq_10min"]).to_numpy(), d["ref_laeq"].to_numpy()
pred = cross_val_predict(LinearRegression(), X, y, cv=KFold(5, shuffle=True, random_state=0))
line = LinearRegression().fit(X, y)
print(f"reference ≈ {line.coef_[0]:.2f} × node {line.intercept_:+.1f} dB")
print("raw node:  ", scores(y, d["laeq_10min"]))
print("calibrated:", scores(y, pred))
plot_vs_reference(y, d["laeq_10min"], pred, f"{SENSOR}: straight line")

❓ Where do the calibrated points still miss the 1:1 line? Think back to notebook 00: what happens at low levels and in wind?

## 1.2 🧪 More Features, Other Models, Two Ways to Split

Two ways to test a model:
- `"random"`: shuffled 10-minute rows, 5 folds. Neighbouring rows are almost the same, so the test is easy.
- `"time"`: fit on the calibration period (September), test on everything after it. This is what happens in a real network.

#### 🚀 Action: one change at a time
1. Add features to `FEATURES` one by one. ❓ Which one helps most?
2. Switch `MODEL` to the random forest. ❓ Why can it handle the noise floor when a straight line cannot?
3. Switch `SPLIT` to `"time"`. ❓ Do the scores hold up? Which node suffers most?
4. Change `SENSOR` in 1.0 and run 1.0 and this cell again.

Each run adds a row to the results table.

In [ ]:
runs = []  # results table: one row per run


def evaluate(model, features, split):
    """Predict with the chosen split and return (y, prediction, raw node reading) for the test rows."""
    if split == "random":
        X, y = d.select(features).to_numpy(), d["ref_laeq"].to_numpy()
        return y, cross_val_predict(model, X, y, cv=KFold(5, shuffle=True, random_state=0)), d["laeq_10min"].to_numpy()
    train, test = d.filter(pl.col("time") < CAL_END), d.filter(pl.col("time") >= CAL_END)
    model.fit(train.select(features).to_numpy(), train["ref_laeq"].to_numpy())
    return test["ref_laeq"].to_numpy(), model.predict(test.select(features).to_numpy()), test["laeq_10min"].to_numpy()

In [ ]:
# 🚀 Action
FEATURES = ["laeq_10min"]  # 🚀 add: "wind_speed", "rain_mm", "temperature", "relative_humidity", "hour"
MODEL = LinearRegression()  # 🚀 try: RandomForestRegressor(n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=0)
SPLIT = "random"  # 🚀 try "time"

y_test, predicted, raw = evaluate(MODEL, FEATURES, SPLIT)
runs.append({"sensor": SENSOR, "model": type(MODEL).__name__, "features": ", ".join(FEATURES), "split": SPLIT}
            | scores(y_test, predicted))
plot_vs_reference(y_test, raw, predicted, f"{SENSOR}: {type(MODEL).__name__}, split = {SPLIT}")
pl.DataFrame(runs)

## 1.3 📆 Does the Calibration Last?

Fit on September only, then score every week after it. A good calibration stays flat. A node that drifts does not.

#### 🚀 Action
- Use your best `FEATURES` and `MODEL` from 1.2.
- ❓ Which node keeps its calibration? Which one loses it slowly, which one suddenly?

##### ▶️ Run

In [ ]:
# 🚀 Action
FEATURES_BEST = ["laeq_10min", "wind_speed", "rain_mm"]
MODEL_BEST = RandomForestRegressor(n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for node in NODES:
    dn = data.filter(pl.col("ID") == node)
    train = dn.filter(pl.col("time") < CAL_END)
    model = MODEL_BEST.fit(train.select(FEATURES_BEST).to_numpy(), train["ref_laeq"].to_numpy())
    dn = dn.with_columns(err=pl.Series(model.predict(dn.select(FEATURES_BEST).to_numpy())) - pl.col("ref_laeq"))
    weekly = dn.group_by_dynamic("time", every="1w").agg(
        pl.col("err").mean().alias("MBE"), (pl.col("err") ** 2).mean().sqrt().alias("RMSE"))
    axes[0].plot(weekly["time"], weekly["MBE"], "o-", color=COLORS[node], label=node)
    axes[1].plot(weekly["time"], weekly["RMSE"], "o-", color=COLORS[node], label=node)
for ax, lab in zip(axes, ["Weekly MBE (dB)", "Weekly RMSE (dB)"]):
    ax.axvline(CAL_END, color="k", ls="--", lw=1); ax.set_ylabel(lab)
axes[0].axhline(0, color="k", lw=0.8); axes[0].legend()
plt.show()

---

# 3 📝 Wrap-up

Note in the template, each with a short reason:

- **Cleaning**: what did you fix or remove before calibrating?
- **Features**: which ones made the difference?
- **Model**: straight line or random forest?
- **Split**: `"random"` or `"time"`, and which scores would you report?
- **Best scores**: your best row of the results table
- **Open question**: what would you do about the nodes that lose their calibration? (Notebook 02)